# Experimento SVM: Predicción Multiclase de Deserción Estudiantil (SPADIES - 3 Clases)
**Autor:** Arnold Santiago Torres | **Machine Learning II (2026-II)**

### Objetivos e Innovaciones Integradas (100% SVM - Norma SPADIES Colombia):
1. **Variable Objetivo Multiclase (SPADIES 3.0)**: Clasificación formal en 3 estados educativos: `Graduado`, `En curso` y `Desertor`, siguiendo el marco del Sistema de Prevención de la Deserción en las Instituciones de Educación Superior (SPADIES - MEN Colombia).
2. **Formulación Transparente del Índice de Riesgo ($R_i \in [0, 7.5]$)**: Índice multidimensional sintético con ponderación explícita de factores académicos, socioeconómicos e institucionales.
3. **Análisis Exploratorio de Datos (EDA) Ampliado**: Diagnóstico de *Outliers* mediante Rango Intercuartílico (IQR), asimetría (skewness) y matrices de correlación de Pearson y Spearman.
4. **Evaluación de Escaladores sobre Variables Sesgadas**: Visualización del impacto de `StandardScaler`, `MinMaxScaler`, `RobustScaler`, `QuantileTransformer` y `PowerTransformer`.
5. **Evaluación Multiclase de SVM sin Fuga de Datos (*Data Leakage*)**: 50 particiones aleatorias independientes (50 trials / 400 modelos entrenados) utilizando `ColumnTransformer` + `Pipeline` de Scikit-Learn.
6. **Inspección del Espacio Latente 2D y Matriz $3 \times 3$**: Visualización del hiperplano de separación vía PCA y auditoría por subgrupos socioeconómicos.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import Parallel, delayed

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    confusion_matrix, classification_report
)
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, MaxAbsScaler, RobustScaler,
    QuantileTransformer, PowerTransformer, OrdinalEncoder, OneHotEncoder, FunctionTransformer
)
from scipy.stats import ttest_ind, skew, kurtosis

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['font.sans-serif'] = 'Arial'

OUTPUT_DIR = 'images'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('✓ Librerías e hiperparámetros multiclase inicializados correctamente.')

--- 
## 1. Origen y Construcción de la Variable Objetivo Multiclase (SPADIES - 3 Clases)

Siguiendo la metodología del **Sistema de Prevención de la Deserción en Educación Superior (SPADIES 3.0)** de Colombia, definimos el estado estudiantil en 3 categorías mutuamente excluyentes ($y \in \{\text{'Desertor'}, \text{'En curso'}, \text{'Graduado'}\}$):

### 1. Índice de Riesgo Ponderado ($R_i \in [0, 7.5]$):
$$R_i = 2.5 \cdot \mathbb{I}(\text{promedio} < 3.0) + 2.5 \cdot \mathbb{I}(\text{reprobadas} \ge 2) + 1.0 \cdot \mathbb{I}(\text{icfes} < 40) + 1.0 \cdot \mathbb{I}(\text{trabaja} = 1) + 0.5 \cdot \mathbb{I}(\text{beca} = 0) + 0.5 \cdot \mathbb{I}(\text{estrato} \le 2)$$

### 2. Reglas de Clasificación SPADIES 3.0:
- **Desertor**: $R_i \ge 3.5$.
- **Graduado**: $R_i < 3.5$ y `semestre_cursado` $\ge$ Duración Nominal del Programa y `promedio_academico` $\ge 3.2$.
- **En curso**: Estudiante activo que no cumple condición de graduado ni de desertor.

In [ ]:
# Carga de datos y construcción de la variable objetivo SPADIES (3 clases)
data_path = 'estudiantes_sinteticos_11500.csv'
if not os.path.exists(data_path):
    data_path = os.path.join('..', 'estudiantes_sinteticos_11500.csv')
if not os.path.exists(data_path):
    data_path = os.path.join('Archivos', 'estudiantes_sinteticos_11500.csv')

df = pd.read_csv(data_path)

riesgo = (
    (df['promedio_academico'] < 3.0).astype(int) * 2.5 +
    (df['materias_reprobadas'] >= 2).astype(int) * 2.5 +
    (df['puntaje_icfes_percentil'] < 40).astype(int) * 1.0 +
    (df['trabaja_mientras_estudia'] == 1).astype(int) * 1.0 +
    (df['beneficiario_beca'] == 0).astype(int) * 0.5 +
    (df['estrato_socioeconomico'] <= 2).astype(int) * 0.5
)
df['riesgo_acumulado'] = riesgo

duracion_map = {'Tecnico profesional': 4, 'Tecnologico': 6, 'Universitario': 10}
df['duracion_nominal'] = df['nivel_formacion'].map(duracion_map).fillna(10)

is_desertor = df['riesgo_acumulado'] >= 3.5
is_graduado = (~is_desertor) & (df['semestre_cursado'] >= df['duracion_nominal']) & (df['promedio_academico'] >= 3.2)

conditions = [is_desertor, is_graduado]
choices = ['Desertor', 'Graduado']
df['estado_final'] = np.select(conditions, choices, default='En curso')

cat_cols = ['sexo', 'estado_civil', 'zona_residencia', 'departamento',
            'nivel_educativo_padre', 'nivel_educativo_madre', 'sector_ies',
            'nivel_formacion', 'metodologia', 'area_conocimiento']

num_cols = ['edad_ingreso', 'promedio_academico', 'materias_reprobadas',
            'distancia_hogar_ies_km', 'puntaje_icfes_percentil', 'semestre_cursado',
            'estrato_socioeconomico', 'anio_registro', 'trabaja_mientras_estudia',
            'beneficiario_icetex', 'beneficiario_beca']

X_raw = df[num_cols + cat_cols]
y = df['estado_final'].values

print('=== DISTRIBUCIÓN DE LA VARIABLE OBJETIVO (3 CLASES - SPADIES) ===')
dist = df['estado_final'].value_counts()
dist_pct = df['estado_final'].value_counts(normalize=True) * 100
dist_df = pd.DataFrame({'Estudiantes': dist, 'Porcentaje': dist_pct.round(2).astype(str) + '%'})
display(dist_df)

# Diagnóstico estadístico de Outliers vía IQR
diag_rows = []
for col in num_cols:
    data_col = df[col].dropna()
    q25, q75 = np.percentile(data_col, [25, 75])
    iqr = q75 - q25
    lower_bound = q25 - 1.5 * iqr
    upper_bound = q75 + 1.5 * iqr
    n_outliers = np.sum((data_col < lower_bound) | (data_col > upper_bound))
    pct_outliers = (n_outliers / len(data_col)) * 100
    sk = skew(data_col)
    diag_rows.append({
        'Variable': col,
        'Media': data_col.mean(),
        'Mediana': data_col.median(),
        'IQR': iqr,
        'Outliers Count': n_outliers,
        'Outliers %': pct_outliers,
        'Skewness': sk
    })

diag_df = pd.DataFrame(diag_rows)
print('=== DIAGNÓSTICO DE OUTLIERS Y ASIMETRÍA (VARIABLES NUMÉRICAS) ===')
display(diag_df.style.format({
    'Media': '{:.2f}', 'Mediana': '{:.2f}', 'IQR': '{:.2f}',
    'Outliers %': '{:.2f}%', 'Skewness': '{:.2f}'
}))

## 2. Análisis Exploratorio de Datos (EDA) y Demostración de Escaladores

In [ ]:
# Gráficos EDA: Distribución de 3 Clases + Correlaciones Pearson vs Spearman
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

palette_3cat = {'En curso': '#2b5c8f', 'Desertor': '#d95f02', 'Graduado': '#2ca02c'}
sns.countplot(data=df, x='estado_final', palette=palette_3cat, ax=axes[0], order=['En curso', 'Desertor', 'Graduado'])
axes[0].set_title('Distribución SPADIES 3 Clases (11,500)', fontweight='bold')
axes[0].set_ylabel('Número de Estudiantes')

# Codificación numérica temporal para correlación (-1: Graduado, 0: En curso, 1: Desertor)
target_code = df['estado_final'].map({'Graduado': -1, 'En curso': 0, 'Desertor': 1})

corr_p = df[num_cols].apply(lambda col: col.corr(target_code, method='pearson')).sort_values()
colors_p = ['#d95f02' if c > 0 else '#7570b3' for c in corr_p.values]
axes[1].barh(corr_p.index, corr_p.values, color=colors_p)
axes[1].set_title('Correlación de Pearson vs Riesgo', fontweight='bold')

corr_s = df[num_cols].apply(lambda col: col.corr(target_code, method='spearman')).sort_values()
colors_s = ['#1b9e77' if c > 0 else '#e7298a' for c in corr_s.values]
axes[2].barh(corr_s.index, corr_s.values, color=colors_s)
axes[2].set_title('Correlación de Spearman vs Riesgo', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Demostración del impacto de escaladores sobre variable sesgada (distancia_hogar_ies_km)
sample_feature = df[['distancia_hogar_ies_km']].values
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

sns.histplot(sample_feature, kde=True, ax=axes[0, 0], color='gray')
axes[0, 0].set_title('1. Original (Raw)', fontweight='bold')

sns.histplot(StandardScaler().fit_transform(sample_feature), kde=True, ax=axes[0, 1], color='blue')
axes[0, 1].set_title('2. Standard Scaler (Z-score)', fontweight='bold')

sns.histplot(MinMaxScaler().fit_transform(sample_feature), kde=True, ax=axes[0, 2], color='green')
axes[0, 2].set_title('3. Min Max Scaler (0 a 1)', fontweight='bold')

sns.histplot(RobustScaler().fit_transform(sample_feature), kde=True, ax=axes[1, 0], color='purple')
axes[1, 0].set_title('4. Robust Scaler (Mediana e IQR)', fontweight='bold')

sns.histplot(QuantileTransformer(n_quantiles=50, random_state=42).fit_transform(sample_feature), kde=True, ax=axes[1, 1], color='orange')
axes[1, 1].set_title('5. Quantile Transformer (Uniforme)', fontweight='bold')

sns.histplot(PowerTransformer().fit_transform(sample_feature), kde=True, ax=axes[1, 2], color='red')
axes[1, 2].set_title('6. Power Transformer (Yeo-Johnson)', fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Experimento de Pipelines SVM Multiclase sin Fuga de Datos (50 Trials / 400 Modelos)

In [ ]:
normalization_methods = {
    'Standard Scaler': StandardScaler(),
    'Min Max Scaler': MinMaxScaler(),
    'Robust Scaler': RobustScaler(),
    'Quantile Transformer': QuantileTransformer(n_quantiles=50, random_state=42, output_distribution='uniform'),
    'Max Absolute Scaler': MaxAbsScaler(),
    'Power Transformer': PowerTransformer(method='yeo-johnson'),
    'Ordinal Encoder': OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1),
    'No Normalization (Baseline)': FunctionTransformer(validate=False)
}

N_TRIALS = 50
N_JOBS = -1

def evaluate_single_trial(trial_idx):
    X_train, X_test, y_train, y_test = train_test_split(
        X_raw, y, test_size=0.2, random_state=42 + trial_idx, stratify=y
    )
    
    trial_macro = {}
    trial_weighted = {}
    trial_acc = {}
    
    for method_name, num_scaler in normalization_methods.items():
        try:
            num_scaler_inst = type(num_scaler)(**num_scaler.get_params()) if hasattr(num_scaler, 'get_params') else num_scaler

            if method_name == 'Ordinal Encoder':
                preprocessor = ColumnTransformer(
                    transformers=[
                        ('num', num_scaler_inst, cat_cols),
                        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), cat_cols)
                    ]
                )
            else:
                preprocessor = ColumnTransformer(
                    transformers=[
                        ('num', num_scaler_inst, num_cols),
                        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), cat_cols)
                    ]
                )

            pipeline = Pipeline(steps=[
                ('preprocessor', preprocessor),
                ('classifier', SVC(kernel='rbf', C=1.0, class_weight='balanced', probability=True, cache_size=1000, random_state=42))
            ])

            pipeline.fit(X_train, y_train)
            preds = pipeline.predict(X_test)
            
            trial_macro[method_name] = f1_score(y_test, preds, average='macro')
            trial_weighted[method_name] = f1_score(y_test, preds, average='weighted')
            trial_acc[method_name] = accuracy_score(y_test, preds)
        except Exception as e:
            trial_macro[method_name] = np.nan
            trial_weighted[method_name] = np.nan
            trial_acc[method_name] = np.nan
            
    return trial_macro, trial_weighted, trial_acc

print(f'Iniciando {N_TRIALS} iteraciones paralelas de Pipelines Multiclase sin fuga de datos...')
parallel_results = Parallel(n_jobs=N_JOBS)(
    delayed(evaluate_single_trial)(t) for t in range(N_TRIALS)
)
print('✓ Iteraciones multiclase completadas exitosamente.')

In [ ]:
# Procesamiento e informe estadístico de los 50 trials multiclase
results_macro = {method: [] for method in normalization_methods.keys()}
results_weighted = {method: [] for method in normalization_methods.keys()}
results_acc = {method: [] for method in normalization_methods.keys()}

for res_mac, res_wei, res_ac in parallel_results:
    for method_name in normalization_methods.keys():
        results_macro[method_name].append(res_mac[method_name])
        results_weighted[method_name].append(res_wei[method_name])
        results_acc[method_name].append(res_ac[method_name])

baseline_scores = results_macro['No Normalization (Baseline)']
summary_rows = []

for method_name in normalization_methods.keys():
    scores_mac = results_macro[method_name]
    scores_wei = results_weighted[method_name]
    scores_ac = results_acc[method_name]
    
    mean_mac = np.mean(scores_mac)
    std_mac = np.std(scores_mac)
    mean_wei = np.mean(scores_wei)
    mean_ac = np.mean(scores_ac)
    
    if method_name != 'No Normalization (Baseline)':
        t_stat, p_val = ttest_ind(scores_mac, baseline_scores, equal_var=False)
        delta = mean_mac - np.mean(baseline_scores)
    else:
        t_stat, p_val, delta = 0.0, 1.0, 0.0
        
    summary_rows.append({
        'Normalization Method': method_name,
        'Macro F1': mean_mac,
        'Weighted F1': mean_wei,
        'Accuracy': mean_ac,
        'Std Dev': std_mac,
        'F1 Delta vs Baseline': delta,
        'T-Statistic': t_stat,
        'P-value vs Baseline': p_val
    })

summary_df = pd.DataFrame(summary_rows).sort_values(by='Macro F1', ascending=False)
display(summary_df.style.format({
    'Macro F1': '{:.4f}', 'Weighted F1': '{:.4f}', 'Accuracy': '{:.4f}',
    'Std Dev': '{:.4f}', 'F1 Delta vs Baseline': '{:+.4f}',
    'T-Statistic': '{:.3f}', 'P-value vs Baseline': '{:.6f}'
}))

# Gráfico comparativo
plt.figure(figsize=(12, 6))
colors_bar = ['#d95f02' if 'Baseline' in name else '#1b9e77' for name in summary_df['Normalization Method']]
barplot = sns.barplot(data=summary_df, x='Macro F1', y='Normalization Method', palette=colors_bar)
plt.title(f'Evaluación Multiclase de Escaladores sobre SVM ({N_TRIALS} Trials / 3 Clases SPADIES)', fontweight='bold')
plt.xlabel('Macro F1-Score Promedio')
plt.xlim(0.60, 0.90)
plt.tight_layout()
plt.show()

## 4. Visualización del Espacio Latente 2D (PCA) y Matriz de Confusión Multiclase ($3 \times 3$)

In [ ]:
# Ajuste del pipeline óptimo con RobustScaler
X_train, X_test, y_train, y_test = train_test_split(X_raw, y, test_size=0.2, random_state=42, stratify=y)

best_preprocessor = ColumnTransformer([
    ('num', RobustScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), cat_cols)
])

best_pipeline = Pipeline([
    ('preprocessor', best_preprocessor),
    ('classifier', SVC(kernel='rbf', C=1.0, class_weight='balanced', probability=True, cache_size=1000, random_state=42))
])

best_pipeline.fit(X_train, y_train)
preds = best_pipeline.predict(X_test)

print('=== REPORTE DE CLASIFICACIÓN MULTICLASE (RobustScaler + SVM RBF) ===')
print(classification_report(y_test, preds, target_names=['Desertor', 'En curso', 'Graduado']))

# Matriz de Confusión 3x3 y Visualización PCA 2D
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

labels = ['Desertor', 'En curso', 'Graduado']
cm = confusion_matrix(y_test, preds, labels=labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=axes[0])
axes[0].set_title('Matriz de Confusión Multiclase (3x3)', fontweight='bold')
axes[0].set_xlabel('Predicción SVM')
axes[0].set_ylabel('Estado Real SPADIES')

# Reducción a 2D vía PCA para visualización de frontera de decisión
X_test_proc = best_pipeline.named_steps['preprocessor'].transform(X_test)
pca = PCA(n_components=2, random_state=42)
X_test_pca = pca.fit_transform(X_test_proc)

# Graficar scatter 2D con fronteras aproximadas
for label, color in zip(['En curso', 'Desertor', 'Graduado'], ['#2b5c8f', '#d95f02', '#2ca02c']):
    mask = (preds == label)
    axes[1].scatter(X_test_pca[mask, 0], X_test_pca[mask, 1], label=f'Pred: {label}', color=color, alpha=0.6, s=20)

axes[1].set_title('Proyección PCA 2D del Espacio Separador SVM', fontweight='bold')
axes[1].set_xlabel('Componente Principal 1')
axes[1].set_ylabel('Componente Principal 2')
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.show()